In [1]:
try:
    import FinanceDataReader
except:
    %pip install FinanceDataReader

In [2]:
import FinanceDataReader as fdr
import pandas as pd

from datetime import datetime, timedelta
from pathlib import Path


# 1. 설정
START = (datetime.now() - timedelta(days=600)).strftime('%Y-%m-%d')
UPDATE_EXISTING = True           # 기존 파일 최신 데이터 갱신

CACHE = Path('stock_cache')
CACHE.mkdir(exist_ok=True)


# 2. KOSPI 종목 목록
list_file = CACHE / 'KOSPI_list.csv'

try:
    stocks = fdr.StockListing('KOSPI')
except Exception as e:
    print('종목 목록 다운로드 실패 → 기존 파일 사용')
    print(e)
    stocks = pd.read_csv(list_file)


# 종목코드 정리
code_col = 'Code' if 'Code' in stocks.columns else 'Symbol'

stocks[code_col] = (
    stocks[code_col].astype(str)
    .str.replace('.0', '', regex=False)
    .str.zfill(6)
)


# 우선주 제외: 삼성전자우, S-Oil우, 현대차2우B 등
stocks = stocks[
    ~stocks['Name'].str.contains(
        r'\d*우[A-Z]?$',
        regex=True,
        na=False
    )
].copy()


# 우선주 제외 후 종목 목록 저장
stocks.to_csv(list_file, index=False)

print('KOSPI 분석 종목 :', len(stocks))


# 3. KOSPI 지수
kospi_file = CACHE / 'KS11.csv'

# 파일이 없으면 전체 다운로드
if not kospi_file.exists():
    kospi = fdr.DataReader('KS11', START)
    kospi.to_csv(kospi_file, index_label='Date')
# 파일이 있으면 읽기
else:
    kospi = pd.read_csv(
        kospi_file,
        index_col='Date',
        parse_dates=['Date']
    )

    # 최신 데이터 갱신
    if UPDATE_EXISTING:
        try:
            start = (
                kospi.index[-1] - pd.Timedelta(days=5)
            ).strftime('%Y-%m-%d')
            new = fdr.DataReader('KS11', start)
            if not new.empty:
                kospi = pd.concat([kospi, new])
                kospi = kospi[
                    ~kospi.index.duplicated(keep='last')
                ].sort_index()
                kospi.to_csv(kospi_file, index_label='Date')
        except Exception as e:
            print('KOSPI 갱신 실패 → 기존 데이터 사용')
            print(e)

# 최신 거래일
market_date = kospi.index[-1].date()

print('최신 거래일 :', market_date)


# 4. 종목별 주가 다운로드
download_count = 0
skip_count = 0
error_count = 0


for n, (_, stock) in enumerate(stocks.iterrows(), 1):
    code = stock[code_col]
    name = stock['Name']
    stock_file = CACHE / f'{code}.csv'
    try:
        # 파일이 없으면 전체 다운로드
        if not stock_file.exists():
            df = fdr.DataReader(code, START)
            if not df.empty:
                df.to_csv(
                    stock_file,
                    index_label='Date'
                )
                download_count += 1

        # 파일이 있으면 필요한 경우만 갱신
        elif UPDATE_EXISTING:
            df = pd.read_csv(
                stock_file,
                index_col='Date',
                parse_dates=['Date']
            )
            last_date = df.index[-1].date()

            # 이미 최신이면 다운로드하지 않음
            if last_date >= market_date:
                skip_count += 1
                continue

            # 부족한 최근 데이터만 다운로드
            start = (
                df.index[-1] - pd.Timedelta(days=5)
            ).strftime('%Y-%m-%d')

            new = fdr.DataReader(
                code,
                start
            )


            if not new.empty:
                df = pd.concat([
                    df,
                    new
                ])

                df = df[
                    ~df.index.duplicated(keep='last')
                ].sort_index()

                df.to_csv(
                    stock_file,
                    index_label='Date'
                )

                download_count += 1

    except Exception as e:
        error_count += 1
        print(f'오류 : {code} {name} → {e}')

    # 진행 상황
    if n % 50 == 0:
        print(f'{n} / {len(stocks)} 종목 확인')


# 5. 결과
print('\n데이터 준비 완료')
print('최신 거래일    :', market_date)
print('다운로드/갱신 :', download_count)
print('최신 파일     :', skip_count)
print('오류          :', error_count)

KOSPI 분석 종목 : 834
최신 거래일 : 2026-08-14

데이터 준비 완료
최신 거래일    : 2026-08-14
다운로드/갱신 : 0
최신 파일     : 834
오류          : 0


In [3]:
# end